# ViT-T + RSA (Attention-Fool)

**Scope:** ViT-Tiny only — RSA `trim` at inference vs six PGD patch attacks.

**ViT-T:** import robust acc from `AttentionFool-2` `results.json` (not re-run here).
**ViT-T-RSA:** set `RUN_RSA_ATTACKS = True` to run attacks on the defended model.


In [ ]:
%pip install torch torchvision
%pip install "timm==1.0.27"


In [ ]:
import os, sys, math, time, json, random
from pathlib import Path
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image
import timm
from timm.data import ImageNetInfo

assert timm.__version__ == '1.0.27', f'expected timm 1.0.27, got {timm.__version__}'

REPO = Path('.').resolve()
sys.path.insert(0, str(REPO))

def _pick_device():
    if torch.cuda.is_available():
        return 'cuda'
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

DEVICE = _pick_device()
print('device:', DEVICE)

## RSA + ViT-T + attack + loader (inline)


In [ ]:
# --- RSA trim defense (inference-time) ---
# Ported from robust-self-attention/pycls/models/vision_transformer.py

def apply_rsa_trim(w, v, *, img_size=224, vit_patch_size=16, adv_patch_size=16):
    """Apply RSA trim to post-softmax weights w and value tokens v."""
    B, H, N, _ = w.shape
    head_dim = v.shape[-1]
    num_patches = img_size // vit_patch_size
    max_patches = adv_patch_size // vit_patch_size + 1
    if adv_patch_size % vit_patch_size > 1:
        max_patches += 1
    context_length = num_patches ** 2 + 1

    v_image = v[:, :, 1:]
    v_mean_head = torch.mean(v_image, dim=2, keepdim=True)
    v_mean_image = torch.mean(v_mean_head, dim=1, keepdim=True).unsqueeze(1)
    v_grid = v_image.reshape(B, H, num_patches, num_patches, head_dim)
    distances = torch.mean(torch.norm(v_grid - v_mean_image, dim=-1), dim=1)
    distances = F.avg_pool2d(distances, max_patches, stride=1)

    idxs = torch.argmax(distances.reshape(B, -1), dim=-1)
    span = num_patches - max_patches + 1
    col_idxs = idxs // span
    row_idxs = idxs % span

    v_mean = v_mean_head.repeat(1, 1, context_length, 1)
    uniform = 1.0 / context_length

    for i in range(max_patches):
        for j in range(max_patches):
            outlier_idx = num_patches * (col_idxs + i) + (row_idxs + j) + 1
            idx_v = outlier_idx.view(B, 1, 1, 1).expand(B, H, context_length, head_dim)
            idx_w = outlier_idx.view(B, 1, 1, 1).expand(B, H, context_length, 1)
            v = v.scatter(-2, idx_v, v_mean)
            w = w.scatter(-1, idx_w, uniform)

    return w, v

print('RSA trim loaded')

In [ ]:
# --- ViT-Tiny with optional RSA trim + pre-softmax QK export ---

def _trunc_normal_(tensor, std=0.02):
    nn.init.trunc_normal_(tensor, std=std)


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=192):
        super().__init__()
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Mlp(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Attention(nn.Module):
    def __init__(self, dim, num_heads=3, qkv_bias=True, attn_drop=0.0, proj_drop=0.0,
                 rsa_trim=False, img_size=224, vit_patch_size=16, adv_patch_size=16):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.rsa_trim = rsa_trim
        self.img_size = img_size
        self.vit_patch_size = vit_patch_size
        self.adv_patch_size = adv_patch_size
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        B_logits = (q @ k.transpose(-2, -1)) * self.scale
        w = B_logits.softmax(dim=-1)
        w = self.attn_drop(w)
        if self.rsa_trim:
            w, v = apply_rsa_trim(w, v, img_size=self.img_size,
                                  vit_patch_size=self.vit_patch_size,
                                  adv_patch_size=self.adv_patch_size)
        x = (w @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x, B_logits


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True, drop=0.0, attn_drop=0.0,
                 rsa_trim=False, img_size=224, vit_patch_size=16, adv_patch_size=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                              attn_drop=attn_drop, proj_drop=drop,
                              rsa_trim=rsa_trim, img_size=img_size,
                              vit_patch_size=vit_patch_size, adv_patch_size=adv_patch_size)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(dim, mlp_ratio=mlp_ratio, drop=drop)

    def forward(self, x):
        attn_out, B_logits = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, B_logits


class VisionTransformer(nn.Module):
    def __init__(self, num_classes=1000, img_size=224, patch_size=16, embed_dim=192,
                 depth=12, num_heads=3, mlp_ratio=4.0, qkv_bias=True, drop_rate=0.0,
                 rsa_trim=False, adv_patch_size=16):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, qkv_bias, drop_rate, drop_rate,
                  rsa_trim=rsa_trim, img_size=img_size, vit_patch_size=patch_size,
                  adv_patch_size=adv_patch_size)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        _trunc_normal_(self.pos_embed, std=0.02)
        _trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            _trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward_features(self, x):
        x = self.patch_embed(x)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        qk_list = []
        for blk in self.blocks:
            x, B = blk(x)
            qk_list.append(B)
        x = self.norm(x)
        return x[:, 0], qk_list

    def forward(self, x):
        feat, qk_list = self.forward_features(x)
        return self.head(feat), qk_list


# ViT-T weights: timm 1.0.27 + augreg_in21k_ft_in1k (matches AttentionFool-2 table)
VIT_T_PRETRAINED = 'vit_tiny_patch16_224.augreg_in21k_ft_in1k'


def vit_tiny_patch16_224(pretrained=True, rsa_trim=False, adv_patch_size=16, num_classes=1000):
    model = VisionTransformer(num_classes=num_classes, rsa_trim=rsa_trim, adv_patch_size=adv_patch_size)
    if pretrained:
        ref = timm.create_model(VIT_T_PRETRAINED, pretrained=True, num_classes=num_classes)
        model.load_state_dict(ref.state_dict(), strict=True)
    return model

print('ViT-T + RSA model loaded')

In [ ]:
# --- Attention-Fool PGD patch attack + robustness eval ---

IMNET_MEAN = (0.485, 0.456, 0.406)
IMNET_STD = (0.229, 0.224, 0.225)
IMNET_MU, IMNET_STD_T = IMNET_MEAN, IMNET_STD


def ce_loss(logits, y, targeted=False, y_target=None):
    if targeted:
        assert y_target is not None
        return -F.cross_entropy(logits, y_target)
    return F.cross_entropy(logits, y)


def single_head_kq_loss(B_hl, target_key_index):
    return B_hl[..., target_key_index].mean()


def l_kq(qk_list, target_key_index):
    layer_scores = []
    for B in qk_list:
        head_scores = torch.stack([single_head_kq_loss(B[:, h], target_key_index)
                                   for h in range(B.shape[1])])
        layer_scores.append(torch.logsumexp(head_scores, dim=0))
    return torch.logsumexp(torch.stack(layer_scores), dim=0)


def single_head_kq_star_loss(B_hl, target_query_index, target_key_index):
    return B_hl[:, target_query_index, target_key_index].mean()


def l_kq_star(qk_list, target_query_index, target_key_index):
    layer_scores = []
    for B in qk_list:
        head_scores = torch.stack([
            single_head_kq_star_loss(B[:, h], target_query_index, target_key_index)
            for h in range(B.shape[1])
        ])
        layer_scores.append(torch.logsumexp(head_scores, dim=0))
    return torch.logsumexp(torch.stack(layer_scores), dim=0)


def build_patch_mask(shape, patch_size, patch_row, patch_col, device):
    mask = torch.zeros(shape, device=device)
    mask[:, :, patch_row:patch_row + patch_size, patch_col:patch_col + patch_size] = 1.0
    return mask


def patch_token_index(patch_row, patch_col, patch_size=16, img_size=224, has_cls_token=True):
    grid = img_size // patch_size
    idx = (patch_row // patch_size) * grid + (patch_col // patch_size)
    return idx + (1 if has_cls_token else 0)


def cosine_step_size(alpha_0, t, n_iters):
    return alpha_0 * 0.5 * (1.0 + math.cos(math.pi * t / n_iters))


def paste_and_normalize(x_norm, patch_01, mask, mu, std):
    x_01 = x_norm * std + mu
    x_01 = patch_01 * mask + x_01 * (1 - mask)
    return (x_01 - mu) / std


def pgd_patch_attack(model, x, y, *,
                     is_transformer=True,
                     loss_type='ce_plus_kq',
                     attn_w=1.0,
                     patch_size=16, patch_row=0, patch_col=0, img_size=224,
                     target_key_index=None, target_query_index=0,
                     n_iters=250, alpha_0=8.0 / 255.0,
                     use_momentum=False, beta=0.9):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    device = x.device
    mu = torch.tensor(IMNET_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(IMNET_STD, device=device).view(1, 3, 1, 1)
    mask = build_patch_mask(x.shape, patch_size, patch_row, patch_col, device)

    if target_key_index is None and is_transformer:
        target_key_index = patch_token_index(patch_row, patch_col, patch_size, img_size)

    x_01 = x * std + mu
    patch_01 = torch.rand_like(x) * mask + x_01 * (1 - mask)
    patch_01 = patch_01.detach().requires_grad_(True)
    momentum = torch.zeros_like(patch_01) if use_momentum else None

    for t in range(n_iters):
        if patch_01.grad is not None:
            patch_01.grad = None

        x_adv = paste_and_normalize(x, patch_01, mask, mu, std)
        if is_transformer:
            logits, qk_list = model(x_adv)
        else:
            logits, qk_list = model(x_adv), None

        loss = ce_loss(logits, y)
        if is_transformer and loss_type == 'ce_plus_kq':
            loss = loss + attn_w * l_kq(qk_list, target_key_index)
        elif is_transformer and loss_type == 'ce_plus_kq_star':
            loss = loss + attn_w * l_kq_star(qk_list, target_query_index, target_key_index)

        grad = torch.autograd.grad(loss, patch_01)[0]
        if use_momentum:
            g = grad / (grad.flatten(1).norm(dim=1).view(-1, 1, 1, 1) + 1e-12)
            momentum = beta * momentum + (1 - beta) * g
            direction = momentum.sign()
        else:
            direction = grad.sign()

        alpha_t = cosine_step_size(alpha_0, t, n_iters)
        with torch.no_grad():
            patch_01 = patch_01 + alpha_t * direction * mask
            patch_01 = patch_01.clamp(0, 1)
            patch_01 = patch_01 * mask + x_01 * (1 - mask)
        patch_01.requires_grad_(True)

    with torch.no_grad():
        x_adv = paste_and_normalize(x, patch_01, mask, mu, std)
    return x_adv, patch_01.detach()


@torch.no_grad()
def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()


def evaluate_robustness(model, loader, device, *,
                        loss_type='ce_plus_kq',
                        n_iters=250,
                        use_momentum=False,
                        max_batches=None):
    model.to(device)
    model.eval()
    clean_correct = robust_correct = total = 0

    for bi, (x, y) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        clean_correct += (logits.argmax(1) == y).sum().item()
        x_adv, _ = pgd_patch_attack(
            model, x, y,
            loss_type=loss_type,
            n_iters=n_iters,
            use_momentum=use_momentum,
        )
        logits_adv, _ = model(x_adv)
        robust_correct += (logits_adv.argmax(1) == y).sum().item()
        total += y.size(0)

    return clean_correct / max(total, 1), robust_correct / max(total, 1)

print('Attention-Fool attack loaded')

In [ ]:
# --- ImageNet val loader (synset folders -> class index) ---

class SynsetImageDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        root = Path(root)
        info = ImageNetInfo('imagenet-1k')
        synset_to_idx = {s: i for i, s in enumerate(info.label_names())}
        self.transform = transform
        self.samples = []
        for syn_dir in sorted(root.iterdir()):
            if not syn_dir.is_dir():
                continue
            label = synset_to_idx.get(syn_dir.name)
            if label is None:
                continue
            for img in sorted(syn_dir.iterdir()):
                if img.suffix.lower() in {'.jpeg', '.jpg', '.png'}:
                    self.samples.append((str(img), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        x = Image.open(path).convert('RGB')
        if self.transform is not None:
            x = self.transform(x)
        return x, label


def make_imagenet_loader(data_dir, img_size=224, batch_size=2, num_workers=0):
    tfm = transforms.Compose([
        transforms.Resize(int(img_size / 0.875), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMNET_MEAN, std=IMNET_STD),
    ])
    ds = SynsetImageDataset(data_dir, transform=tfm)
    if len(ds) == 0:
        raise FileNotFoundError(f'No images found under {data_dir}')
    print(f'SynsetImageDataset: {len(ds)} images from {data_dir}')
    return torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=False,
    )

print('ImageNet loader loaded')

## Config


In [ ]:
ATTACK_ITERS = 250
SEED = 0
PAPER_ATTACKS = [
    ('pgd_lce', 'ce', False),
    ('pgd_lce_lkq', 'ce_plus_kq', False),
    ('pgd_lce_lkqstar', 'ce_plus_kq_star', False),
    ('pgd_lce_mom', 'ce', True),
    ('pgd_lce_mom_lkq', 'ce_plus_kq', True),
    ('pgd_lce_mom_lkqstar', 'ce_plus_kq_star', True),
]
RUN_RSA_ATTACKS = True
RSA_BATCH_SIZE = 4
RSA_MAX_BATCHES = None
BASELINE_RESULTS_PATH = REPO / 'results.json'
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)


## ViT-T / ViT-T-RSA


In [ ]:
def _vit_t(rsa_trim=False):
    return vit_tiny_patch16_224(pretrained=True, rsa_trim=rsa_trim), True, 224
MODEL_FACTORY = {'ViT-T': lambda: _vit_t(False), 'ViT-T-RSA': lambda: _vit_t(True)}


## Data


In [ ]:
_data_candidates = [
    REPO / 'data' / 'imagenet_val',
    Path('/workspace/Attention-Fool/data/imagenet_val'),
]
IMAGENET_DIR = next((p for p in _data_candidates if p.is_dir() and any(p.iterdir())), None)
if IMAGENET_DIR is None:
    raise FileNotFoundError('Missing imagenet_val — use Attention-Fool/scripts/prepare_demo_data.py')
print('images:', sum(1 for _ in IMAGENET_DIR.rglob('*.[jJ][pP]*')))


## Eval


In [ ]:
def load_vit_t_baseline(path=BASELINE_RESULTS_PATH):
    if not Path(path).is_file():
        print('No results.json — copy from AttentionFool-2'); return {}
    data = json.loads(Path(path).read_text())
    return {tag: {'clean_acc': data[tag]['ViT-T']['clean_acc'],
                  'robust_acc': data[tag]['ViT-T'].get('attnfool_acc', data[tag]['ViT-T'].get('robust_acc'))}
            for tag, _, _ in PAPER_ATTACKS if tag in data and 'ViT-T' in data[tag]}

def eval_vit_t_rsa_all_attacks():
    loader = make_imagenet_loader(IMAGENET_DIR, batch_size=RSA_BATCH_SIZE)
    model = vit_tiny_patch16_224(pretrained=True, rsa_trim=True)
    out = {}
    for tag, lt, mom in PAPER_ATTACKS:
        c, r = evaluate_robustness(model, loader, DEVICE, loss_type=lt, n_iters=ATTACK_ITERS,
                                   use_momentum=mom, max_batches=RSA_MAX_BATCHES)
        out[tag] = {'clean_acc': c, 'robust_acc': r}
        print(tag, f'clean={100*c:.1f}% robust={100*r:.1f}%')
    return out

print('ViT-T baseline:', json.dumps(load_vit_t_baseline(), indent=2))
if RUN_RSA_ATTACKS:
    p = REPO / 'results_vit_t_rsa.json'
    p.write_text(json.dumps({'ViT-T': load_vit_t_baseline(), 'ViT-T-RSA': eval_vit_t_rsa_all_attacks()}, indent=2))
    print('Wrote', p)
else:
    print('RUN_RSA_ATTACKS=False')
